## Week 5 - Assignment 5 for COfDSAI Tutorial Markdown ##

This week looks at audit selection classfication while taking into account potential discrimination.

We want to wrangle the features a bit to be able to extract more information from them.

1. Setup, Feature Engineering, and Convex Estimator Functions

Start by importing the required packages and defining the data-preprocessing and optimization helpers:
- **Feature engineering (`build_features`):** Builds 38 standardized features (numeric variables, quadratic terms `age_sq` and `hours_sq`, a missing-job flag, sex, and one-hot encoded categoricals with the most frequent level dropped as reference and levels below 1% merged into `RARE`). Excludes `race` (unless `with_race=True`), `native-country`, and `fnlwgt`.
- **Convex estimator (`fit`):** Solves $\ell_1$-regularized logistic regression (splitting $w = p - q$, $p, q \ge 0$ for smooth SLSQP optimization) with an optional smoothed-hinge surrogate FPR cap ($\kappa$) per race group and/or a mean-score constraint ($\epsilon$) for comparison.
- **Evaluation helpers (`surrogate_fpr`, `best_thr`, `make_groups`):** Computes per-group surrogate FPR values, selects the score threshold minimizing training error, and maps dummy columns back to variable-level groups.

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from load_adult import load_adult, race_group, GROUPS, key_numbers, group_table

# 1. CONFIGURATION
STUDENT = "i6194134"
CATS = ["workclass", "marital-status", "occupation", "relationship"]
RARE = 0.01                                         # Merge categorical levels below 1% into "RARE"
TAU = 0.1                                           # Softplus smoothing parameter for surrogate FPR cap
SEEDS = range(5)                                    # 5 repeated 5-fold CV splits to reduce tuning noise


# 2. FEATURE ENGINEERING
def build_features(tr, te, with_race=False):
    # Numeric + squares + missing flag + one-hot (reference = most frequent, <1% merged into RARE).
    # Standardised using training mean and standard deviation only.
    def raw(df):
        D = pd.DataFrame(index=df.index)
        D["age"] = df["age"]
        D["education-num"] = df["education-num"]
        D["hours"] = df["hours-per-week"]
        D["cap-gain"] = np.log1p(df["capital-gain"])
        D["cap-loss"] = np.log1p(df["capital-loss"])
        D["missing_job"] = (df["workclass"].isna() | df["occupation"].isna()).astype(float)
        D["male"] = (df["sex"] == "Male").astype(float)
        return D

    Dtr, Dte = raw(tr), raw(te)

    for c in CATS + (["race"] if with_race else []):
        vc = tr[c].value_counts(normalize=True)
        keep = [v for v in vc.index if vc[v] >= RARE]
        ref = vc.index[0]
        merged = [v for v in vc.index if vc[v] < RARE]
        for df, D in ((tr, Dtr), (te, Dte)):
            # NaN stays NaN (all dummies 0; missing_job flags it)
            s = df[c].where(df[c].isna() | df[c].isin(keep), "RARE")
            for v in [v for v in keep if v != ref] + (["RARE"] if merged else []):
                D[f"{c}={v}"] = (s == v).astype(float)

    # Add standardized quadratic terms for age and hours
    ma, sa = Dtr["age"].mean(), Dtr["age"].std()
    mh, sh = Dtr["hours"].mean(), Dtr["hours"].std()
    for D in (Dtr, Dte):
        D["age_sq"] = ((D["age"] - ma) / sa) ** 2
        D["hours_sq"] = ((D["hours"] - mh) / sh) ** 2

    mu, sd = Dtr.mean(), Dtr.std() + 1e-9
    return ((Dtr - mu) / sd).values, ((Dte - mu) / sd).values, list(Dtr.columns)


def make_groups(cols):
    # Map one-hot dummies and quadratic terms back to their parent variable name
    key = lambda c: c.split("=")[0] if "=" in c else {"age_sq": "age", "hours_sq": "hours"}.get(c, c)
    names = list(dict.fromkeys(key(c) for c in cols))
    return names, [np.array([i for i, c in enumerate(cols) if key(c) == n]) for n in names]


# 3. CONVEX ESTIMATOR (SLSQP) & EVALUATION HELPERS
def fit(X, y, g, lam, eps, penalty="l1", groups=None, kappa=np.inf, tau=TAU):
    # Minimize mean logistic loss + lam * P(w) subject to optional fairness constraints:
    #   - Mean-score constraint (eps < inf): |(mu_g - mu_0)' w| <= eps for each race group g (y=0)
    #   - Surrogate FPR cap (kappa < inf):   mean_{i in N_g} softplus_tau(1 + w'x_i + b) <= kappa
    # For l1, split w = p - q (p, q >= 0) so the objective is smooth for SLSQP.
    n, d = X.shape
    s = 2 * y - 1
    neg = y == 0
    mu0 = X[neg].mean(0)
    C = np.array([X[neg & (g == k)].mean(0) - mu0 for k in GROUPS])
    split = penalty == "l1"

    def f(z):
        w = z[:d] - z[d : 2 * d] if split else z[:d]
        m = s * (X @ w + z[-1])
        r = -s * expit(-m) / n
        gw = X.T @ r
        val = np.mean(np.logaddexp(0, -m))
        if penalty == "l1":
            val += lam * z[: 2 * d].sum()
            gz = np.concatenate([gw + lam, -gw + lam])
        elif penalty == "l2":
            val += lam * (w @ w)
            gz = gw + 2 * lam * w
        else:
            gp = np.zeros(d)
            for idx in groups:
                nr = np.sqrt(w[idx] @ w[idx] + 1e-8)
                a = lam * np.sqrt(len(idx))
                val += a * nr
                gp[idx] = a * w[idx] / nr
            gz = gw + gp
        return val, np.concatenate([gz, [r.sum()]])

    cons = []
    if np.isfinite(eps):
        A = np.hstack([C, -C, np.zeros((len(GROUPS), 1))]) if split else np.hstack([C, np.zeros((len(GROUPS), 1))])
        cons.append({
            "type": "ineq",
            "fun": lambda z: np.concatenate([eps - A @ z, eps + A @ z]),
            "jac": lambda z: np.vstack([-A, A]),
        })

    if np.isfinite(kappa):
        Xn = [X[neg & (g == k)] for k in GROUPS]

        def cf(z):
            w = z[:d] - z[d : 2 * d] if split else z[:d]
            return np.array([kappa - np.mean(tau * np.logaddexp(0, (1 + Xk @ w + z[-1]) / tau)) for Xk in Xn])

        def cj(z):
            w = z[:d] - z[d : 2 * d] if split else z[:d]
            J = []
            for Xk in Xn:
                sg = expit((1 + Xk @ w + z[-1]) / tau)
                gw = Xk.T @ sg / len(sg)
                J.append(-np.concatenate([gw, -gw, [sg.mean()]]) if split else -np.concatenate([gw, [sg.mean()]]))
            return np.array(J)

        cons.append({"type": "ineq", "fun": cf, "jac": cj})

    m_ = 2 * d if split else d
    bounds = [(0, None)] * m_ + [(None, None)] if split else [(None, None)] * (m_ + 1)
    r = minimize(f, np.zeros(m_ + 1), jac=True, method="SLSQP", constraints=cons,
                 bounds=bounds, options={"maxiter": 600, "ftol": 1e-10})

    w = r.x[:d] - r.x[d : 2 * d] if split else r.x[:d].copy()
    if penalty == "group":
        for idx in groups:
            if np.linalg.norm(w[idx]) < 1e-3:
                w[idx] = 0
    else:
        w[np.abs(w) < 1e-6] = 0
    return w, r.x[-1], r.status, np.abs(C @ w).max()


def surrogate_fpr(X, y, g, w, b, tau=TAU):
    # Per-group mean of softplus_tau(1 + score) over people with y=0 (the quantity capped by kappa)
    return {k: float(np.mean(tau * np.logaddexp(0, (1 + X[(y == 0) & (g == k)] @ w + b) / tau))) for k in GROUPS}


def best_thr(sc, y):
    # Grid search for the score threshold minimizing training classification error
    ts = np.linspace(-3, 3, 241)
    return ts[int(np.argmin([np.mean((sc > t) != y) for t in ts]))]

2. Hyperparameter Tuning ($\lambda$ and $\kappa$) via Cross-Validation

Using strictly the 32,561 training records (the test set is never used for parameter selection), this cell:
- **Tunes $\lambda$:** Runs 5-fold CV on logistic log-loss and selects the largest $\lambda$ within `0.0005` of the minimum for sparsity ($\lambda = 10^{-3}$).
- **Scales the burden cap $\kappa$:** Computes the unconstrained baseline's largest group surrogate FPR ($\kappa_{\max} \approx 0.281$) and sweeps fractions $f \in \{\infty, 0.8, 0.65, 0.5, 0.4, 0.3, 0.2\}$ over 5 repeated 5-fold CV splits for both race-excluded (recommended) and race-as-input (reference) models.
- **Compares against the mean-score alternative:** Evaluates the mean-score constraint $|(\mu_g - \mu_0)^\top w| \le \epsilon$ across $\epsilon \in \{\infty, 0.3, 0.2, 0.1, 0.05, 0.02\}$ on the same CV splits.

In [ ]:
T0 = time.time()
tr, te = load_adult("adult.csv")
ytr = (tr["income"] == ">50K").values.astype(int)
yte = (te["income"] == ">50K").values.astype(int)
gtr = race_group(tr["race"])

Xtr, Xte, cols = build_features(tr, te)

# --- 1. Select lambda by 5-fold CV log-loss (largest lambda within 0.0005 of best) ---
skf0 = StratifiedKFold(5, shuffle=True, random_state=0)
ll = {}
for lam in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]:
    v = []
    for a, b in skf0.split(Xtr, ytr):
        w, bb, st, _ = fit(Xtr[a], ytr[a], gtr[a], lam, np.inf)
        v.append(np.mean(np.logaddexp(0, -(2 * ytr[b] - 1) * (Xtr[b] @ w + bb))))
    ll[lam] = np.mean(v)
    print(f"lambda {lam} | CV logloss {ll[lam]:.4f}")

LAM = max(l for l in ll if ll[l] <= min(ll.values()) + 0.0005)
print(f"Chosen lambda: {LAM}\n")

# --- 2. Baseline surrogate scale (kappa_max) & CV sweep grids ---
w0, b0, st0, _ = fit(Xtr, ytr, gtr, LAM, np.inf)
S0 = surrogate_fpr(Xtr, ytr, gtr, w0, b0)
KM = max(S0.values())
print("Baseline surrogate per group:", {k: round(v, 3) for k, v in S0.items()}, "| kappa_max:", round(KM, 3))

FR = [np.inf, 0.8, 0.65, 0.5, 0.4, 0.3, 0.2]       # Cap fractions: kappa = frac * kappa_max
EPS_REF = [np.inf, 0.3, 0.2, 0.1, 0.05, 0.02]      # Mean-score constraint grid (comparison only)


def evaluate(X1, X2, eps, kap):
    # Average out-of-fold metrics over 5 repeated 5-fold CV splits, then fit on full training set
    cvs, nfit = [], 0
    for sd in SEEDS:
        oof = np.zeros(len(ytr), bool)
        for a, b in StratifiedKFold(5, shuffle=True, random_state=sd).split(X1, ytr):
            w, bb, st, _ = fit(X1[a], ytr[a], gtr[a], LAM, eps, kappa=kap)
            assert st == 0
            nfit += 1
            oof[b] = (X1[b] @ w + bb) > best_thr(X1[a] @ w + bb, ytr[a])
        k = key_numbers(ytr == 1, oof, tr["race"])
        gt = group_table(ytr == 1, oof, tr["race"])
        k["max_fpr"] = gt.loc[GROUPS, "fpr"].max()
        k["tpr_all"] = gt.loc["All", "tpr"]
        cvs.append(k)

    cv = {key: float(np.mean([c[key] for c in cvs])) for key in cvs[0]}
    w, bb, st, _ = fit(X1, ytr, gtr, LAM, eps, kappa=kap)
    assert st == 0
    nfit += 1
    thr = best_thr(X1 @ w + bb, ytr)
    sel = (X2 @ w + bb) > thr
    return dict(
        cv=cv, te=key_numbers(yte == 1, sel, te["race"]), sel=sel,
        nnz=int((w != 0).sum()), w=w, b=bb, thr=thr, nfit=nfit,
        surr=surrogate_fpr(X1, ytr, gtr, w, bb) if np.isfinite(kap) else None,
    )


# --- 3. Run CV sweeps for surrogate cap (race-free & race-input) and mean-score constraint ---
res, nfits = {}, 0
for wr in [False, True]:
    X1, X2, _ = build_features(tr, te, with_race=wr)
    tag = "race_input" if wr else "race-free"
    for f in FR:
        res[(wr, f)] = evaluate(X1, X2, np.inf, f * KM)
        nfits += res[(wr, f)]["nfit"]
        print(f"{tag} | kappa frac {f} | CV { {k: round(v, 2) for k, v in res[(wr, f)]['cv'].items()} } | nnz {res[(wr, f)]['nnz']}", flush=True)

X1, X2, _ = build_features(tr, te)
ref = {}
for e in EPS_REF:
    if np.isfinite(e):
        ref[e] = evaluate(X1, X2, e, np.inf)
        nfits += ref[e]["nfit"]
    else:
        ref[e] = res[(False, np.inf)]
    print(f"mean-constraint | eps {e} | CV { {k: round(v, 2) for k, v in ref[e]['cv'].items()} }", flush=True)

print(f"\nTotal SLSQP fits (all status 0): {nfits}")

3. Model Selection, Test Evaluation, and Solver Verification

This cell applies the training-set selection rule (smallest CV FPR gap with CV error at most 0.5 pp above baseline) to pick $f = 0.4$ ($\kappa \approx 0.112$) and $\epsilon = 0.1$, then:
- Evaluates the baseline, recommended capped model, mean-score alternative, and race-as-input reference on the held-out test set (printing `group_table` for Table 1).
- Cross-checks the unconstrained SLSQP objective against scikit-learn's `liblinear` solver.
- Re-solves both the unconstrained baseline and the final capped model directly in **CVXPY (`CLARABEL`, exponential cones)** to verify convexity, conic optimality (`status = optimal`), and identical objectives and test metrics.

In [ ]:
# --- 1. Select kappa and eps strictly from training CV ---
base = res[(False, np.inf)]["cv"]
FRC = min([f for f in FR if res[(False, f)]["cv"]["error"] <= base["error"] + 0.5],
          key=lambda f: res[(False, f)]["cv"]["fpr_gap"])
EPSC = min([e for e in EPS_REF if ref[e]["cv"]["error"] <= base["error"] + 0.5],
           key=lambda e: ref[e]["cv"]["fpr_gap"])

print(f"Chosen kappa fraction: {FRC} (kappa = {FRC * KM:.5f}) | Chosen eps (reference): {EPSC}")
B, F, M = res[(False, np.inf)], res[(False, FRC)], ref[EPSC]

print("BASE test:", B["te"])
print("SURR test:", F["te"], "| nnz:", F["nnz"], "| cutoff:", round(F["thr"], 3),
      "| group surrogate:", {k: round(v, 3) for k, v in F["surr"].items()})
print("MEAN-CONSTRAINT test:", M["te"], "| CV:", M["cv"])
print("SURR CV:", F["cv"])
print("BASE CV:", B["cv"])

for nm, r in (("Baseline", B), ("Surrogate cap (recommended)", F), ("Mean constraint (alternative)", M)):
    print(f"\n{nm} — Test Group Table:")
    print(group_table(yte == 1, r["sel"], te["race"]).round(1).to_string())

print("\nBaseline with race as input (test):", res[(True, np.inf)]["te"])
print("Race-input surrogate, same kappa (test):", res[(True, FRC)]["te"], "| CV:", res[(True, FRC)]["cv"])
print(group_table(yte == 1, res[(True, FRC)]["sel"], te["race"]).round(1).to_string())

# --- 2. Unconstrained solver check against scikit-learn liblinear ---
s_tr = 2 * ytr - 1
obj = lambda w, b: np.mean(np.logaddexp(0, -s_tr * (Xtr @ w + b))) + LAM * np.abs(w).sum()
sk = LogisticRegression(penalty="l1", C=1 / (LAM * len(ytr)), solver="liblinear", tol=1e-10, max_iter=5000).fit(Xtr, ytr)
b_sk = minimize_scalar(lambda b: obj(sk.coef_[0], b)).x
print(f"\nUnconstrained objective -> SLSQP: {obj(w0, b0):.6f} | liblinear: {obj(sk.coef_[0], b_sk):.6f}")

# --- 3. Conic optimality verification in CVXPY (CLARABEL, exponential cones) ---
def fit_cvx(kappa=None):
    n, d = Xtr.shape
    w, b = cp.Variable(d), cp.Variable()
    z = Xtr @ w + b
    loss = cp.sum(cp.logistic(-cp.multiply(s_tr, z))) / n
    cons = []
    if kappa is not None:
        for k in GROUPS:
            idx = np.where((ytr == 0) & (gtr == k))[0]
            cons.append(cp.sum(TAU * cp.logistic((1 + z[idx]) / TAU)) / len(idx) <= kappa)
    prob = cp.Problem(cp.Minimize(loss + LAM * cp.norm1(w)), cons)
    prob.solve(solver="CLARABEL", verbose=False)
    return w.value, b.value, prob


def compare_cvx(kappa, label):
    w1, b1, _, _ = fit(Xtr, ytr, gtr, LAM, np.inf, kappa=kappa if kappa else np.inf)
    w2, b2, prob = fit_cvx(kappa)
    sel = lambda w, b: (Xte @ w + b) > best_thr(Xtr @ w + b, ytr)
    t1 = key_numbers(yte == 1, sel(w1, b1), te["race"])
    t2 = key_numbers(yte == 1, sel(w2, b2), te["race"])
    print(f"\n--- CVXPY Check: {label} (status: {prob.status}) ---")
    print(f"Objective    -> SLSQP: {obj(w1, b1):.5f} | CVXPY: {obj(w2, b2):.5f}")
    print(f"Max |w diff| -> {np.abs(w1 - w2).max():.4f} | nnz (|w|>1e-4): SLSQP {(np.abs(w1) > 1e-4).sum()} vs CVXPY {(np.abs(w2) > 1e-4).sum()}")
    print(f"Test metrics -> err {t1['error']:.2f} vs {t2['error']:.2f} | FPR gap {t1['fpr_gap']:.2f} vs {t2['fpr_gap']:.2f} | TPR gap {t1['tpr_gap']:.2f} vs {t2['tpr_gap']:.2f}")


compare_cvx(None, "Baseline (no cap)")
compare_cvx(FRC * KM, f"Surrogate cap (kappa = {FRC}*KM = {FRC * KM:.3f})")
compare_cvx(0.8 * KM, f"Loosest cap check (kappa = 0.8*KM = {0.8 * KM:.3f})")

4. Bootstrap Feature-Selection Stability Check

As a robustness check on the recommended capped model ($\kappa = 0.4 \cdot \kappa_{\max}$), this cell draws `B = 30` non-parametric bootstrap resamples of the training set, refits the model on each resample, and reports:
- Column-level selection frequency (`|w| > 1e-3`) and sign agreement.
- Variable-level selection frequency (whether at least one dummy/term of each of the 11 parent variables is selected).

In [ ]:
t0 = time.time()
B_BOOT, TOL = 30, 1e-3
KAPPA = FRC * KM
rng = np.random.default_rng(0)

sel_boot, wts_boot = [], []
for _ in range(B_BOOT):
    i = rng.choice(len(ytr), len(ytr), replace=True)
    w, _, st, _ = fit(Xtr[i], ytr[i], gtr[i], LAM, np.inf, kappa=KAPPA)
    assert st == 0
    sel_boot.append(np.abs(w) > TOL)
    wts_boot.append(w)

sel_boot, wts_boot = np.array(sel_boot), np.array(wts_boot)
freq = sel_boot.mean(0)


def sign_agree(k):
    nz = wts_boot[sel_boot[:, k], k]
    return 100 * max((nz > 0).mean(), (nz < 0).mean()) if len(nz) else np.nan


table = pd.DataFrame({
    "column": cols,
    "selected_%": (100 * freq).round(0),
    "status": np.where(freq >= 0.9, "stable (in)", np.where(freq <= 0.1, "stable (out)", "UNSTABLE")),
    "sign_agree_%": [round(sign_agree(k)) if sel_boot[:, k].any() else np.nan for k in range(len(cols))],
    "mean_coef": wts_boot.mean(0).round(3),
}).sort_values("selected_%")

print(f"{B_BOOT} bootstrap refits (kappa={KAPPA:.3f}) took {time.time() - t0:.0f}s\n")
print("Columns by selection frequency (least stable first):")
print(table.to_string(index=False))
print("\nSUMMARY:", (table.status == "stable (in)").sum(), "stable (in) |",
      (table.status == "stable (out)").sum(), "stable (out) |",
      (table.status == "UNSTABLE").sum(), "unstable:", list(table.loc[table.status == "UNSTABLE", "column"]))

names, G = make_groups(cols)
vtab = pd.DataFrame({"variable": names, "selected_%": [round(100 * sel_boot[:, idx].any(1).mean()) for idx in G]})
print("\nVariable-level selection frequency:")
print(vtab.to_string(index=False))

lambda 0.0001 CV logloss 0.3235
lambda 0.0003 CV logloss 0.3236
lambda 0.001 CV logloss 0.3239
lambda 0.003 CV logloss 0.3264
lambda 0.01 CV logloss 0.3396
chosen lambda 0.001


5. Trade-Off Figure and `keynumbers.csv` Export

This cell generates the 3-panel trade-off figure (`tradeoff_surrogate.pdf` — error, FPR gap, and DP/TPR gaps across $\kappa$) and exports the required `keynumbers_i6194134_surrogate.csv` file matching Appendix A.

In [ ]:
# --- 1. Trade-off figure (tradeoff_surrogate.pdf) ---
ix = list(range(len(FR)))
lab = ["none"] + [str(f) for f in FR[1:]]

fig, ax = plt.subplots(1, 3, figsize=(10.5, 2.7))
for wr, c, nm in [(False, "C0", "race not an input"), (True, "C1", "race an input")]:
    for ax_, key in zip(ax[:2], ["error", "fpr_gap"]):
        ax_.plot(ix, [res[(wr, f)]["cv"][key] for f in FR], "o-", c=c, label=nm + " (train CV)")
        ax_.plot(ix, [res[(wr, f)]["te"][key] for f in FR], "s:", c=c, alpha=0.6, label=nm + " (test)")
    ax[2].plot(ix, [res[(wr, f)]["te"]["tpr_gap"] for f in FR], "s-", c=c, label=nm + " TPR gap")
    ax[2].plot(ix, [res[(wr, f)]["te"]["dp_gap"] for f in FR], "^--", c=c, label=nm + " DP gap")

for a_, t_ in zip(ax, ["error (%)", "FPR gap (pp)", "other gaps, test (pp)"]):
    a_.set_xticks(ix)
    a_.set_xticklabels(lab, fontsize=7)
    a_.set_xlabel(r"cap $\kappa$ as share of baseline maximum", fontsize=8)
    a_.set_title(t_, fontsize=9)
    a_.tick_params(labelsize=7)

ax[0].legend(fontsize=5.5)
ax[2].legend(fontsize=5.5)
for a_ in ax[:2]:
    a_.axvline(FR.index(FRC), c="k", lw=0.5)

plt.tight_layout()
plt.savefig("tradeoff_surrogate.pdf")
plt.close(fig)

# --- 2. Export keynumbers CSV (Appendix A) ---
f2 = lambda v: f"{v:.2f}"
k = {
    "student_number": STUDENT,
    "model": f"L1-regularised logistic regression (lambda={LAM}) with a convex cap kappa={FRC * KM:.3f} on the smoothed hinge surrogate of the false positive rate of each of the four race groups; threshold minimising training error",
    "fairness": "no race group bears more unjustified checks than a common cap: surrogate false positive rate of every group among people with income <=50K bounded by kappa; race only used in the constraints",
    "problem_class": "exponential cone",
    "race_used": "no",
    "n_train": len(tr),
    "n_features": F["nnz"],
    "test_error": f2(F["te"]["error"]),
    "sel_rate": f2(F["te"]["sel_rate"]),
    "dp_gap": f2(F["te"]["dp_gap"]),
    "fpr_gap": f2(F["te"]["fpr_gap"]),
    "tpr_gap": f2(F["te"]["tpr_gap"]),
    "base_test_error": f2(B["te"]["error"]),
    "base_dp_gap": f2(B["te"]["dp_gap"]),
    "base_fpr_gap": f2(B["te"]["fpr_gap"]),
}

out_csv = f"keynumbers_{STUDENT}_surrogate.csv"
pd.DataFrame({"key": list(k), "value": list(k.values())}).to_csv(out_csv, index=False)
print(open(out_csv).read())
print("Total elapsed seconds:", round(time.time() - T0))

baseline surrogate per group {'White': 0.187, 'Black': 0.074, 'Asian-Pac-Islander': 0.281, 'Other': 0.109} kappa_max 0.281
race-free kappa frac inf CV {'error': 15.21, 'sel_rate': 20.47, 'dp_gap': 18.6, 'fpr_gap': 9.85, 'tpr_gap': 22.34, 'max_fpr': 13.03, 'tpr_all': 60.94} nnz 37
race-free kappa frac 0.8 CV {'error': 15.22, 'sel_rate': 20.71, 'dp_gap': 17.72, 'fpr_gap': 9.38, 'tpr_gap': 20.55, 'max_fpr': 12.53, 'tpr_all': 61.4} nnz 38
race-free kappa frac 0.65 CV {'error': 15.29, 'sel_rate': 20.67, 'dp_gap': 16.87, 'fpr_gap': 8.6, 'tpr_gap': 19.65, 'max_fpr': 11.8, 'tpr_all': 61.16} nnz 36
race-free kappa frac 0.5 CV {'error': 15.46, 'sel_rate': 20.31, 'dp_gap': 16.15, 'fpr_gap': 8.0, 'tpr_gap': 19.44, 'max_fpr': 11.25, 'tpr_all': 60.08} nnz 38
race-free kappa frac 0.4 CV {'error': 15.54, 'sel_rate': 19.62, 'dp_gap': 15.86, 'fpr_gap': 7.41, 'tpr_gap': 19.84, 'max_fpr': 10.56, 'tpr_all': 58.46} nnz 37
race-free kappa frac 0.3 CV {'error': 15.75, 'sel_rate': 20.22, 'dp_gap': 15.43, 'fpr_

In [16]:
# ---- choice of kappa on the training set only: smallest CV FPR gap with CV error <= baseline + 0.5 pp
base = res[(False, np.inf)]["cv"]
FRC = min([f for f in FR if res[(False, f)]["cv"]["error"] <= base["error"]+0.5], key=lambda f: res[(False, f)]["cv"]["fpr_gap"])
EPSC = min([e for e in EPS_REF if ref[e]["cv"]["error"] <= base["error"]+0.5], key=lambda e: ref[e]["cv"]["fpr_gap"])
print("chosen kappa fraction", FRC, "kappa", FRC*KM, "| chosen eps (mean constraint, reference)", EPSC)
B, F, M = res[(False, np.inf)], res[(False, FRC)], ref[EPSC]
print("BASE test", B["te"]); print("SURR test", F["te"], "nnz", F["nnz"], "cutoff", F["thr"], "group surrogate", {k: round(v, 3) for k, v in F["surr"].items()}, "kappa", FRC*KM)
print("MEAN-CONSTRAINT test", M["te"], "CV", M["cv"]); print("SURR CV", F["cv"]); print("BASE CV", B["cv"])
for nm, r in (("baseline", B), ("surrogate", F), ("mean constraint", M)): print("\n", nm); print(group_table(yte == 1, r["sel"], te["race"]).round(1).to_string())
print("\nbaseline with race as input, test:", res[(True, np.inf)]["te"])
print("race-input surrogate, same kappa, test:", res[(True, FRC)]["te"], "CV", res[(True, FRC)]["cv"]); print(group_table(yte == 1, res[(True, FRC)]["sel"], te["race"]).round(1).to_string())
gtab = group_table(ytr == 1, (Xtr@F["w"]+F["b"]) > F["thr"], tr["race"]); print("train FPR by group at chosen model:", gtab["fpr"].round(2).to_dict())
print("train positives share", ytr.mean(), "| test", yte.mean(), "| group sizes train", pd.Series(gtr).value_counts().to_dict())

chosen kappa fraction 0.4 kappa 0.11233976561716286 | chosen eps (mean constraint, reference) 0.1
BASE test {'error': 14.863951845709721, 'sel_rate': 21.178060315705423, 'dp_gap': 18.660046978432625, 'fpr_gap': 9.015561959654178, 'tpr_gap': 16.88311688311689}
SURR test {'error': 15.22633744855967, 'sel_rate': 19.194152693323506, 'dp_gap': 14.589205637411915, 'fpr_gap': 6.357348703170029, 'tpr_gap': 13.909774436090231} nnz 37 cutoff -0.5999999999999996 group surrogate {'White': 0.087, 'Black': 0.032, 'Asian-Pac-Islander': 0.112, 'Other': 0.056} kappa 0.11233976561716286
MEAN-CONSTRAINT test {'error': 15.502733247343528, 'sel_rate': 18.131564400221116, 'dp_gap': 14.982993197278912, 'fpr_gap': 7.286455331412103, 'tpr_gap': 15.447710184552292} CV {'error': 15.650624980805258, 'sel_rate': 19.251865728939528, 'dp_gap': 16.503014728012992, 'fpr_gap': 7.5805977106582905, 'tpr_gap': 18.612497030173444, 'max_fpr': 11.743119266055047, 'tpr_all': 57.47736258130341}
SURR CV {'error': 15.54497711986

Compare the objective value of the solver to check it makes sense.

In [17]:
# ---- solver check against liblinear (unconstrained)
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
obj = lambda w, b: np.mean(np.logaddexp(0, -(2*ytr-1)*(Xtr@w+b)))+LAM*np.abs(w).sum()
sk = LogisticRegression(penalty="l1", C=1/(LAM*len(ytr)), solver="liblinear", tol=1e-10, max_iter=5000).fit(Xtr, ytr)
print("objective mine", obj(w0, b0), "liblinear", obj(sk.coef_[0], minimize_scalar(lambda b: obj(sk.coef_[0], b)).x))

objective mine 0.3311100025586814 liblinear 0.3311198973125949


In [18]:
# ---- figure
ix = list(range(len(FR))); lab = ["none"]+[str(f) for f in FR[1:]]
fig, ax = plt.subplots(1, 3, figsize=(10.5, 2.7))
for wr, c, nm in [(False, "C0", "race not an input"), (True, "C1", "race an input")]:
    for ax_, key in zip(ax[:2], ["error", "fpr_gap"]):
        ax_.plot(ix, [res[(wr, f)]["cv"][key] for f in FR], "o-", c=c, label=nm+" (train CV)")
        ax_.plot(ix, [res[(wr, f)]["te"][key] for f in FR], "s:", c=c, alpha=.6, label=nm+" (test)")
    ax[2].plot(ix, [res[(wr, f)]["te"]["tpr_gap"] for f in FR], "s-", c=c, label=nm+" TPR gap")
    ax[2].plot(ix, [res[(wr, f)]["te"]["dp_gap"] for f in FR], "^--", c=c, label=nm+" DP gap")
for a_, t_ in zip(ax, ["error (%)", "FPR gap (pp)", "other gaps, test (pp)"]):
    a_.set_xticks(ix); a_.set_xticklabels(lab, fontsize=7); a_.set_xlabel(r"cap $\kappa$ as share of baseline maximum", fontsize=8); a_.set_title(t_, fontsize=9); a_.tick_params(labelsize=7)
ax[0].legend(fontsize=5.5); ax[2].legend(fontsize=5.5)
for a_ in ax[:2]: a_.axvline(FR.index(FRC), c="k", lw=.5)
plt.tight_layout(); plt.savefig("tradeoff_surrogate.pdf")

# ---- key numbers
f2 = lambda v: f"{v:.2f}"
k = {"student_number": "i6194134",
    "model": f"L1-regularised logistic regression (lambda={LAM}) with a convex cap kappa={FRC*KM:.3f} on the smoothed hinge surrogate of the false positive rate of each of the four race groups; threshold minimising training error",
    "fairness": "no race group bears more unjustified checks than a common cap: surrogate false positive rate of every group among people with income <=50K bounded by kappa; race only used in the constraints",
    "problem_class": "exponential cone", "race_used": "no", "n_train": len(tr), "n_features": F["nnz"],
    "test_error": f2(F["te"]["error"]), "sel_rate": f2(F["te"]["sel_rate"]), "dp_gap": f2(F["te"]["dp_gap"]),
    "fpr_gap": f2(F["te"]["fpr_gap"]), "tpr_gap": f2(F["te"]["tpr_gap"]),
    "base_test_error": f2(B["te"]["error"]), "base_dp_gap": f2(B["te"]["dp_gap"]), "base_fpr_gap": f2(B["te"]["fpr_gap"])}
pd.DataFrame({"key": list(k), "value": list(k.values())}).to_csv("keynumbers_i6194134_surrogate.csv", index=False)
print(open("keynumbers_i6194134_surrogate.csv").read()); print("total seconds", round(time.time()-T0))

key,value
student_number,i6194134
model,L1-regularised logistic regression (lambda=0.001) with a convex cap kappa=0.112 on the smoothed hinge surrogate of the false positive rate of each of the four race groups; threshold minimising training error
fairness,no race group bears more unjustified checks than a common cap: surrogate false positive rate of every group among people with income <=50K bounded by kappa; race only used in the constraints
problem_class,exponential cone
race_used,no
n_train,32561
n_features,37
test_error,15.23
sel_rate,19.19
dp_gap,14.59
fpr_gap,6.36
tpr_gap,13.91
base_test_error,14.86
base_dp_gap,18.66
base_fpr_gap,9.02

total seconds 427


Confirm using CVXPY that this formulation is convex and optimal. 

In [ ]:
tr, te = load_adult("adult.csv")
ytr = (tr["income"] == ">50K").values.astype(int); yte = (te["income"] == ">50K").values.astype(int)
gtr = race_group(tr["race"])
X, Xte, cols = build_features(tr, te)
n, d = X.shape; s = 2*ytr - 1; LAM, TAU = 1e-3, 0.1

def fit_cvx(kappa=None):
    w, b = cp.Variable(d), cp.Variable()
    z = X @ w + b
    loss = cp.sum(cp.logistic(-cp.multiply(s, z))) / n             # mean log(1+exp(-s*z))
    cons = []
    if kappa is not None:                                           # one cap per race group
        for k in GROUPS:
            idx = np.where((ytr == 0) & (gtr == k))[0]
            cons.append(cp.sum(TAU*cp.logistic((1 + z[idx])/TAU)) / len(idx) <= kappa)
    prob = cp.Problem(cp.Minimize(loss + LAM*cp.norm1(w)), cons)
    prob.solve(verbose = True)               
    print(f"cvxpy status: {prob.status}")                                     # exponential cone solver (cvxpy picks one)
    return w.value, b.value, prob

obj = lambda w, b: np.mean(np.logaddexp(0, -s*(X@w + b))) + LAM*np.abs(w).sum()

def compare(kappa, label):
    w1, b1, st1, _ = fit(X, ytr, gtr, LAM, np.inf, kappa=kappa if kappa else np.inf)   # my SLSQP
    w2, b2, prob = fit_cvx(kappa)                                                      # cvxpy
    sel = lambda w, b: (Xte@w + b) > best_thr(X@w + b, ytr)
    t1 = key_numbers(yte == 1, sel(w1, b1), te["race"]); t2 = key_numbers(yte == 1, sel(w2, b2), te["race"])
    print(f"--- {label}: cvxpy status {prob.status}")
    print("objective  SLSQP %.5f | cvxpy %.5f" % (obj(w1, b1), obj(w2, b2)))
    print("max |w diff| %.4f | non-zero (|w|>1e-4)  SLSQP %d | cvxpy %d" % (np.abs(w1-w2).max(), (np.abs(w1) > 1e-4).sum(), (np.abs(w2) > 1e-4).sum()))
    print("test error %.2f vs %.2f | FPR gap %.2f vs %.2f | TPR gap %.2f vs %.2f" %
          (t1["error"], t2["error"], t1["fpr_gap"], t2["fpr_gap"], t1["tpr_gap"], t2["tpr_gap"]))

compare(None, "baseline (no cap)")
w0, b0, _, _ = fit(X, ytr, gtr, LAM, np.inf)
KM = max(surrogate_fpr(X, ytr, gtr, w0, b0).values())               # 0.281 in my run
compare(0.4*KM, "surrogate cap, kappa = 0.4*KM (about 0.112)")

(CVXPY) Nov 26 01:50:02 PM: Your problem has 39 variables, 0 constraints, and 0 parameters.
(CVXPY) Nov 26 01:50:02 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Nov 26 01:50:02 PM: DCP verification time: 0.0025 seconds.
(CVXPY) Nov 26 01:50:02 PM: Expression tree has 9 nodes.
(CVXPY) Nov 26 01:50:02 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Nov 26 01:50:02 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Nov 26 01:50:02 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Nov 26 01:50:02 PM: Compiling problem (target solver=CLARABEL).
(CVXPY) Nov 26 01:50:02 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> EliminateZeroSized -> ConeMatrixStuffing -> CLARABEL
(CVXPY) Nov 26 01:50:02 PM: Applying reduction Dcp2Cone
(CVXPY) Nov 26 01:50:02 PM: Applying reduction CvxAttr2Constr
(CVXPY) Nov 26 01:50:02 PM: Ap

                                     CVXPY                                     
                                     v1.9.3                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Nov 26 01:50:03 PM: Applying reduction CLARABEL
(CVXPY) Nov 26 01:50:04 PM: Finished problem compilation (took 1.794e+00 seconds).
(CVXPY) Nov 26 01:50:04 PM: Invoking solver CLARABEL  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 97760
  constraints   = 228003
  nnz(P)        = 0
  nnz(A)        = 1465397
  cones (total) = 65123
    : Nonnegative = 1,  numel = 32637
    : Exponential = 65122,  numel = (3,3,3,3,...,3)

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-8, tol_gap_abs = 1.0e-8, tol_gap_rel = 1.0e-8,
  static reg : on, ϵ1 = 1.0e-8, ϵ

(CVXPY) Nov 26 01:50:25 PM: Problem status: optimal
(CVXPY) Nov 26 01:50:25 PM: Optimal value: 3.311e-01
(CVXPY) Nov 26 01:50:25 PM: Compilation took 1.794e+00 seconds
(CVXPY) Nov 26 01:50:25 PM: Solver (including time spent in interface) took 2.070e+01 seconds


 24  +3.3111e-01  +3.3111e-01  8.86e-09  1.53e-11  1.81e-11  9.93e-14  1.83e-13  6.34e-01  
---------------------------------------------------------------------------------------------
Terminated with status = Solved
solve time = 20.2963037s
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
cvxpy status: optimal
--- baseline (no cap): cvxpy status optimal
objective  SLSQP 0.33111 | cvxpy 0.33111
max |w diff| 0.0001 | non-zero (|w|>1e-4)  SLSQP 37 | cvxpy 37
test error 14.86 vs 14.87 | FPR gap 9.02 vs 9.02 | TPR gap 16.88 vs 16.88


... (2 outputs left out)

(CVXPY) Nov 26 01:50:28 PM: Applying reduction CLARABEL
(CVXPY) Nov 26 01:50:29 PM: Finished problem compilation (took 3.231e+00 seconds).
(CVXPY) Nov 26 01:50:29 PM: Invoking solver CLARABEL  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
-------------------------------------------------------------
           Clarabel.rs v0.11.1  -  Clever Acronym                

                   (c) Paul Goulart                          
                University of Oxford, 2022                   
-------------------------------------------------------------

problem:
  variables     = 171920
  constraints   = 401047
  nnz(P)        = 0
  nnz(A)        = 2602517
  cones (total) = 114563
    : Nonnegative = 1,  numel = 57361
    : Exponential = 114562,  numel = (3,3,3,3,...,3)

settings:
  linear algebra: direct / qdldl, precision: 64 bit (1 thread)
  max iter = 200, time limit = Inf,  max step = 0.990
  tol_feas = 1.0e-8, tol_gap_abs = 1.0e-8, tol_gap_rel = 1.0e-8,
  static reg : on, ϵ1 = 1.0e-8

(CVXPY) Nov 26 01:51:51 PM: Problem status: optimal
(CVXPY) Nov 26 01:51:51 PM: Optimal value: 3.541e-01
(CVXPY) Nov 26 01:51:51 PM: Compilation took 3.231e+00 seconds
(CVXPY) Nov 26 01:51:51 PM: Solver (including time spent in interface) took 8.205e+01 seconds


 55  +3.5412e-01  +3.5412e-01  9.86e-09  5.96e-14  6.60e-11  2.37e-13  9.11e-14  2.60e-01  
---------------------------------------------------------------------------------------------
Terminated with status = Solved
solve time = 81.6120562s
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------
cvxpy status: optimal
--- surrogate cap, kappa = 0.4*KM (about 0.112): cvxpy status optimal
objective  SLSQP 0.35412 | cvxpy 0.35412
max |w diff| 0.0000 | non-zero (|w|>1e-4)  SLSQP 37 | cvxpy 37
test error 15.23 vs 15.23 | FPR gap 6.36 vs 6.36 | TPR gap 13.91 vs 13.91


Just for a sanity check I look at the stability of the chosen features as well.

In [20]:
# ---- Bootstrap stability of the selected features (final surrogate model) ----
# Question: would the model pick the same columns if the training sample were different?
# Method: refit the capped model on B bootstrap resamples and record which columns get a non-zero coefficient.
import time; t0 = time.time()
tr, te = load_adult("adult.csv")
ytr = (tr["income"] == ">50K").values.astype(int)        # label: 1 = income >50K
gtr = race_group(tr["race"])                              # race group per person (used only inside the constraint)
X, Xte, cols = build_features(tr, te); LAM = 1e-3         # standardised features (fixed once, not redone per resample)

# Cap kappa: 0.4 times the largest group surrogate value of the unconstrained model (as chosen in the main analysis)
w0, b0, _, _ = fit(X, ytr, gtr, LAM, np.inf)
KM = max(surrogate_fpr(X, ytr, gtr, w0, b0).values()); KAPPA = 0.4*KM   # about 0.112

B, TOL = 30, 1e-3          # B = number of bootstrap resamples; a column counts as "selected" if |coefficient| > TOL
rng = np.random.default_rng(0)                            # fixed seed -> the same 30 resamples every run
sel, wts = [], []
for _ in range(B):
    # Bootstrap resample: draw n = len(ytr) = 32,561 row indices, each uniformly at random from ALL rows,
    # putting each row back after it is drawn (replace=True). Some rows appear several times, about 37% not at all.
    i = rng.choice(len(ytr), len(ytr), replace=True)
    # The same index list i picks features, labels and race groups, so each person's data stays together.
    w, b, st, _ = fit(X[i], ytr[i], gtr[i], LAM, np.inf, kappa=KAPPA); assert st == 0   # solver status 0 = converged
    sel.append(np.abs(w) > TOL)                           # which columns have a non-zero coefficient in this resample
    wts.append(w)                                         # the coefficients themselves (for sign checks)
sel, wts = np.array(sel), np.array(wts)                   # shape (B, number of columns)
freq = sel.mean(0)                                        # share of resamples in which each column was selected

def sign_agree(k):
    """Among resamples that selected column k: percentage with the majority sign (100 = always the same sign)."""
    nz = wts[sel[:, k], k]
    return 100*max((nz > 0).mean(), (nz < 0).mean()) if len(nz) else np.nan

# Status rule: selected in >=90% of resamples -> stable (in); <=10% -> stable (out); otherwise UNSTABLE
table = pd.DataFrame({"column": cols, "selected_%": (100*freq).round(0),
                      "status": np.where(freq >= .9, "stable (in)", np.where(freq <= .1, "stable (out)", "UNSTABLE")),
                      "sign_agree_%": [round(sign_agree(k)) if sel[:, k].any() else np.nan for k in range(len(cols))],
                      "mean_coef": wts.mean(0).round(3)}).sort_values("selected_%")
print(f"{B} bootstrap refits of the capped model (kappa={KAPPA:.3f}) took {time.time()-t0:.0f} s\n")
print("Columns by selection frequency (least stable first):"); print(table.to_string(index=False))
print("\nSUMMARY:", (table.status == "stable (in)").sum(), "always selected |", (table.status == "stable (out)").sum(), "never selected |",
      (table.status == "UNSTABLE").sum(), "unstable:", list(table.loc[table.status == "UNSTABLE", "column"]))

# Variable level: a variable (e.g. occupation, made of many dummies) counts as selected if ANY of its dummies is selected
names, G = make_groups(cols)
vtab = pd.DataFrame({"variable": names, "selected_%": [round(100*sel[:, idx].any(1).mean()) for idx in G]})
print("\nVariable-level selection frequency:"); print(vtab.to_string(index=False))

30 bootstrap refits of the capped model (kappa=0.112) took 21 s

Columns by selection frequency (least stable first):
                              column  selected_%      status  sign_agree_%  mean_coef
          relationship=Not-in-family        60.0    UNSTABLE           100     -0.128
          occupation=Protective-serv        73.0    UNSTABLE            91      0.023
             occupation=Tech-support        80.0    UNSTABLE            58     -0.011
                 workclass=Local-gov        83.0    UNSTABLE            92      0.029
                    occupation=Sales        83.0    UNSTABLE           100     -0.044
               workclass=Federal-gov        87.0    UNSTABLE            85      0.018
             marital-status=Divorced        90.0 stable (in)            96     -0.249
              workclass=Self-emp-inc        93.0 stable (in)            71      0.024
            marital-status=Separated        93.0 stable (in)           100     -0.141
          occupation=E

In [ ]:
gtab = group_table(ytr == 1, (Xtr @ F["w"] + F["b"]) > F["thr"], tr["race"])
print("Train FPR by group at chosen model:", gtab["fpr"].round(2).to_dict())
print("Train positives share:", round(ytr.mean(), 4), "| Test:", round(yte.mean(), 4),
      "| Train group sizes:", pd.Series(gtr).value_counts().to_dict())